# L2 step 1c · The options rerun: the new 7B answers with the options shown, for option (c) (GPU T4 ×2)

Task **L2** (roadmap), the distilled 3B for objective **O4**. Decided 8 Nov: train two 3Bs, **(b)** with soft labels and **(c)** with this rerun, and keep the better one by the rule in roadmap L2, fixed before any run.

**Why:** L2 step 1 asked the 7B the plain training question, without the options, and kept only answers that matched the label. Counting and relational reasoning kept few (16.3% and 15.6%), because the 7B had to guess an exact number or sentence. With the five options shown, as in the benchmark, it only has to pick one, so more answers can be kept.

**The known risk, and the guard.** On the benchmark's counting questions whose largest option is above 20, the options are built around the correct count, so the middle value is always right. Examples kept here show the 3B those options, so (c)'s 3B could learn that pattern. The rule in roadmap L2 therefore judges counting only on held-out questions whose largest option is 20 or less, and reports how often each 3B picks the middle option.

**What it does**
- The same questions as L2 step 1's first run (4,000 drawn with seed 42), restricted to building counting (1,445), road counting (735) and relational reasoning (180), minus those on held-out after-photos.
- The benchmark's own prompts (H1 V2's and L0 step 2's), with the options; full-size images, as in the benchmark; greedy; the 499A scorer's letter check. Kept: answers whose letters equal the label's.

**Expectation, fixed before the run (8 Nov):** kept shares near the 7B's benchmark accuracy at full size (building counting 30.3%, road counting 33.0%, relational 34.6%), or higher, since these are training questions the 7B has seen: about 700–900 kept. Time: about 1.5 h.

**How to run on Kaggle** (account 1)
1. Settings: GPU **T4 x2**, internet ON. No secret is needed.
2. Input: `abrarmohammedtanzim/disasterm3-mirror`.
3. Run All once with `SMOKE_TEST = True` in Cell 2 (16 questions). Then set it to `False` and use Save & Run All.
4. Outputs in `/kaggle/working/L2S1c/`: `options_kept.jsonl` (the kept examples for (c)'s training), `options_progress.jsonl` (every answer), `by_task.csv`, `README.md`, `config.json`.


In [ ]:
# ── Cell 1: Environment (Kaggle's own torch; transformers only if too old for Qwen2.5-VL) ──
import importlib.metadata as _md
import subprocess
import sys

def _version(pkg):
    try:
        return _md.version(pkg)
    except _md.PackageNotFoundError:
        return "0"

def _older(have, want):
    nums = lambda v: tuple(int(x) for x in v.split(".")[:2] if x.isdigit())
    return nums(have) < nums(want)

if _older(_version("transformers"), "4.49") or _version("accelerate") == "0":
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers>=4.49", "accelerate"], check=True)

import torch
import transformers
print(f"✓ torch {torch.__version__}, transformers {transformers.__version__}, python {sys.version.split()[0]}")
print(f"  GPUs: {[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]}")
assert torch.cuda.device_count() >= 2, "Pick GPU T4 x2: the 7B in fp16 needs both T4s"


In [ ]:
# ── Cell 2: Configuration ─────────────────────────────────────────────────
import glob, json, math, os, random, re, shutil, time
from pathlib import Path

from PIL import Image

DATA_ROOT = Path("/kaggle/input/datasets/abrarmohammedtanzim/disasterm3-mirror/DisasterM3_Instruct")
MANIFEST_PATH = DATA_ROOT / "train_release.json"
TEACHER_MODEL_ID = "AbrarAlam/disasterm3-qwen2.5vl7b-l0s2-mergedFP"
GEN_MAX_NEW_TOKENS = 8         # the answer is one letter
BATCH = 8
SMOKE_TEST = False             # True = 16 questions
TIME_LIMIT_HOURS = 11.0
FIRST_RUN_SIZE, RANDOM_SEED, FIRST_RUN_CHECKABLE = 4000, 42, 40233     # L2 step 1's draw, reproduced exactly
RERUN = {"Building Damage Counting": 1445, "Road Damage Counting": 735, "relational reasoning": 180}   # in that draw
BENCHMARK_ACCURACY = {"Building Damage Counting": 30.31, "Road Damage Counting": 33.03, "relational reasoning": 34.62}  # L0 step 2
OUTPUT_DIR = Path("/kaggle/working/L2S1c")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PROGRESS_PATH = OUTPUT_DIR / "options_progress.jsonl"

def hf_token():
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        return os.environ.get("HF_TOKEN")

# ── The rules all L2 step 2 notebooks share (the same text in each) ─────────
import hashlib

PATH_SUFFIXES = (".png", ".jpg", ".jpeg", ".tif", ".tiff")
CHECKABLE = {"Building Damage Counting", "Road Damage Counting", "Disaster Type Recognition",
             "Disaster Bearing Bodies Recognition", "Disaster Scene Recognition", "relational reasoning"}
MAX_PIXELS = 768 * 28 * 28       # the 3B's image cap: a 1024 px image reaches it at 756 px (729 tokens); also the soft labels'
HELD_OUT_EVERY = 20              # 1 after-photo in 20 is held out to choose between the two 3Bs (fixed 8 Nov)
ELIGIBLE, CHECKABLE_TRAIN, CHECKABLE_HELD_OUT = 55764, 38271, 1962   # counted on the PC from train_release.json, 8 Nov

def name_of(path):
    return Path(str(path).replace("\\", "/")).name

def scene(e):
    """The after-photo a question is about. A relational question's boxed picture, '<after-photo>_<k>.png',
    counts as its after-photo, so the same scene never sits on both sides of the split."""
    if e.get("post_image_path"):
        return name_of(e["post_image_path"])
    return re.sub(r"_\d+(\.png)$", r"\1", name_of(e.get("image_path", "")))

def held_out(e):
    return int(hashlib.md5(scene(e).encode()).hexdigest(), 16) % HELD_OUT_EVERY == 0

def resolve_image_path(rel_path):
    """L2 step 1's rule: a manifest path -> its file in the mirror (None if missing)."""
    if not rel_path:
        return None
    rel_path = rel_path.replace("\\", "/")
    filename = Path(rel_path).name
    for candidate in (DATA_ROOT / rel_path, DATA_ROOT / "train_images" / rel_path, DATA_ROOT / "box_train_images" / rel_path,
                      DATA_ROOT / "masks" / rel_path, DATA_ROOT / "train_images" / "train_images" / filename,
                      DATA_ROOT / "box_train_images" / "box_train_images" / filename, DATA_ROOT / "masks" / "masks" / filename,
                      Path("/tmp") / rel_path):
        if candidate.exists():
            return str(candidate)
    return None

def answer_text(entry):
    ans = entry.get("training_answer", "")
    if isinstance(ans, list):
        ans = ans[0] if ans else ""
    return str(ans).strip()

def eligible(entry):
    """L2 step 1's rule (the 7B's training rule): no segmentation, a text answer, and every image found."""
    if entry.get("task") == "Referring Expression Segmentation":
        return False
    ans = answer_text(entry)
    if not ans or ans.lower().endswith(PATH_SUFFIXES):
        return False
    if "image_path" in entry:
        return resolve_image_path(entry["image_path"]) is not None
    return (resolve_image_path(entry.get("pre_image_path", "")) is not None
            and resolve_image_path(entry.get("post_image_path", "")) is not None)

def smart_resize(height, width, min_pixels, max_pixels, factor=28):
    """Qwen2.5-VL's rule for the size an image reaches the model at (its image processor's smart_resize)."""
    h, w = round(height / factor) * factor, round(width / factor) * factor
    if h * w > max_pixels:
        beta = math.sqrt(height * width / max_pixels)
        h, w = math.floor(height / beta / factor) * factor, math.floor(width / beta / factor) * factor
    elif h * w < min_pixels:
        beta = math.sqrt(min_pixels / (height * width))
        h, w = math.ceil(height * beta / factor) * factor, math.ceil(width * beta / factor) * factor
    return h, w

def sized(img, max_pixels=MAX_PIXELS):
    """The image at the 3B's cap: only an image above it is scaled down (as L0 did at full size)."""
    if img.width * img.height <= max_pixels:
        return img
    h, w = smart_resize(img.height, img.width, 56 * 56, max_pixels)
    return img.resize((w, h), Image.BICUBIC)

assert smart_resize(1024, 1024, 56 * 56, MAX_PIXELS) == (756, 756)
assert held_out({"image_path": "x\\box_train_images\\a_post_7_3.png"}) == held_out({"post_image_path": "a_post_7.png"})

# The benchmark's prompts, copied from L0 step 2 (which copied them from the DisasterM3 authors' runner)
SINGLE_CHOICE = """Analyze both the pre-disaster and post-disaster images to answer the following question. Choose the best option from the candidate options provided.

pre-disaster image:
<image>

post-disaster image:
<image>

Question: {prompt}
Options: {options_str}

Your task is to respond with ONLY the capital letter of the correct option (e.g., C). Do not include any explanation or other text."""
RELATIONAL = """Analyze the image to answer the following question. Choose the best option from the candidate options provided.

Question: {prompt}
Options: {options_str}

Your task is to respond with ONLY the capital letter of the correct option (e.g., C). Do not include any explanation or other text."""

def letters(text):
    """The 499A scorer: the set of capital letters left after removing 'THE ANSWER IS' and 'OPTION'."""
    t = str(text).upper().replace("THE ANSWER IS", "").replace("OPTION", "")
    return set(re.findall(r"[A-Z]", t))

assert letters("B") == {"B"} and letters("The answer is D.") == {"D"} and letters("C, D.") == {"C", "D"}
print(f"Teacher: {TEACHER_MODEL_ID}; the options shown; greedy" + ("  [SMOKE TEST: 16 questions]" if SMOKE_TEST else ""))


In [ ]:
# ── Cell 3: Load the 7B in fp16 across both T4s, and check the full size ─────
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration

processor = AutoProcessor.from_pretrained(TEACHER_MODEL_ID, token=hf_token())
processor.tokenizer.padding_side = "left"             # batched generation needs left padding
dtype_arg = "torch_dtype" if _older(transformers.__version__, "4.56") else "dtype"
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(TEACHER_MODEL_ID, **{dtype_arg: torch.float16}, device_map="auto",
                                                           attn_implementation="sdpa", token=hf_token())
model.eval()
_, h, w = processor.image_processor(images=[Image.new("RGB", (1024, 1024), (90, 120, 80))], return_tensors="pt")["image_grid_thw"][0].tolist()
assert (h * 14, w * 14) == (1036, 1036), f"The 7B would see {h * 14} x {w * 14} px, not 1036 (the benchmark's full size)"
print("✓ 7B loaded; full size checked (1036 px)")


In [ ]:
# ── Cell 4: L2 step 1's draw, its counting and relational questions, the benchmark prompts ──
with open(MANIFEST_PATH, encoding="utf-8") as f:
    raw_data = json.load(f)
eligible_entries = [e for e in raw_data if eligible(e)]
checkable = [i for i, e in enumerate(eligible_entries) if e.get("task") in CHECKABLE]
assert (len(eligible_entries), len(checkable)) == (ELIGIBLE, FIRST_RUN_CHECKABLE), "not L2 step 1's pool"
random.seed(RANDOM_SEED)
first_run = set(random.sample(checkable, k=FIRST_RUN_SIZE))
in_draw = {t: sum(eligible_entries[i]["task"] == t for i in first_run) for t in RERUN}
assert in_draw == RERUN, f"not L2 step 1's draw: {in_draw}"
TODO_ALL = sorted(i for i in first_run if eligible_entries[i]["task"] in RERUN and not held_out(eligible_entries[i]))
if SMOKE_TEST:
    TODO_ALL = sorted(random.Random(RANDOM_SEED).sample(TODO_ALL, 16))

def prompt_of(entry):
    """The benchmark's prompt for this question, with '<image>' where each picture goes, and the image keys in order."""
    prompt = entry.get("prompts", "")
    prompt = prompt[0] if isinstance(prompt, list) else prompt
    if entry["task"] == "relational reasoning":            # its entries spell the options' field option_str
        return "<image>" + RELATIONAL.format(prompt=prompt, options_str=entry["option_str"]), ["image_path"]
    return SINGLE_CHOICE.format(prompt=prompt, options_str=entry["options_str"]), ["pre_image_path", "post_image_path"]

def content_of(entry):
    """As the benchmark runner builds it: the text split at each '<image>', a picture slot in each gap."""
    text, keys = prompt_of(entry)
    parts = text.split("<image>")
    content = []
    for n, part in enumerate(parts):
        if part:
            content.append({"type": "text", "text": part})
        if n < len(parts) - 1:
            content.append({"type": "image"})
    return content, keys

def messages_of(entry):
    """The prompt and its pictures, at full size, in the order the slots appear."""
    content, keys = content_of(entry)
    return [{"role": "user", "content": content}], [Image.open(resolve_image_path(entry[k])).convert("RGB") for k in keys]

for _i in TODO_ALL:                                       # every prompt has one slot per picture of its question
    _content, _keys = content_of(eligible_entries[_i])
    assert sum(c["type"] == "image" for c in _content) == len(_keys), f"entry {_i}: slots and pictures differ"
print(f"✓ {len(TODO_ALL):,} questions (held-out after-photos left out): "
      f"{ {t: sum(eligible_entries[i]['task'] == t for i in TODO_ALL) for t in RERUN} }")


In [ ]:
# ── Cell 5: Answers (greedy, batched, resumable, written as they come) ────────
from tqdm.auto import tqdm

previous = sorted(glob.glob("/kaggle/input/**/L2S1c/options_progress.jsonl", recursive=True))
if previous and not PROGRESS_PATH.exists():
    shutil.copy(previous[0], PROGRESS_PATH)
done = set()
if PROGRESS_PATH.exists():
    with open(PROGRESS_PATH, encoding="utf-8") as f:
        done = {json.loads(line)["_idx"] for line in f if line.strip()}
todo = [i for i in TODO_ALL if i not in done]
print(f"{len(done):,} already answered; {len(todo):,} to do")

@torch.inference_mode()
def generate(idxs):
    convs = [messages_of(eligible_entries[i]) for i in idxs]
    texts = [processor.apply_chat_template(m, tokenize=False, add_generation_prompt=True) for m, _ in convs]
    inputs = processor(text=texts, images=[im for _, ims in convs for im in ims], padding=True, return_tensors="pt").to(model.device)
    out = model.generate(**inputs, max_new_tokens=GEN_MAX_NEW_TOKENS, do_sample=False, temperature=None, top_p=None, top_k=None,
                         repetition_penalty=1.0)
    return [s.strip() for s in processor.batch_decode(out[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True)]

def generate_safe(idxs):
    try:
        return generate(idxs)
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        if len(idxs) == 1:
            return [""]
        half = len(idxs) // 2
        return generate_safe(idxs[:half]) + generate_safe(idxs[half:])

deadline, t0, stopped, shown = time.time() + TIME_LIMIT_HOURS * 3600, time.time(), False, 0
with open(PROGRESS_PATH, "a", encoding="utf-8") as fh:
    for start in tqdm(range(0, len(todo), BATCH), desc="Answers"):
        if time.time() > deadline:
            stopped = True
            break
        idxs = todo[start:start + BATCH]
        for i, answer in zip(idxs, generate_safe(idxs)):
            e = eligible_entries[i]
            kept = bool(answer) and letters(answer) == letters(e["ground_truth_option"])
            fh.write(json.dumps({"_idx": i, "task": e["task"], "answer": answer, "label": e["ground_truth_option"], "kept": kept}) + "\n")
            if shown < 4:
                print(f"[{e['task']}] label {e['ground_truth_option']!r}, answer {answer!r}: {'kept' if kept else 'not kept'}")
                shown += 1
        fh.flush()
RUN_HOURS = (time.time() - t0) / 3600
print(f"{'Time limit reached: stopped cleanly' if stopped else 'Pass complete'} after {RUN_HOURS:.2f} h")


In [ ]:
# ── Cell 6: The kept examples for (c)'s training, by task, and the output files ──
import pandas as pd

with open(PROGRESS_PATH, encoding="utf-8") as f:
    rows = [json.loads(line) for line in f if line.strip()]
kept_rows = [r for r in rows if r["kept"]]
with open(OUTPUT_DIR / "options_kept.jsonl", "w", encoding="utf-8") as fh:
    for r in kept_rows:
        e = eligible_entries[r["_idx"]]
        text, keys = prompt_of(e)
        fh.write(json.dumps({"entry_idx": r["_idx"], "task": e["task"], "prompt": text,
                             "images": [e[k] for k in keys], "answer": "".join(sorted(letters(r["answer"])))}) + "\n")
by_task = pd.DataFrame([{"task": t, "answered": sum(r["task"] == t for r in rows), "kept": sum(r["task"] == t and r["kept"] for r in rows)}
                        for t in RERUN])
by_task["kept_%"] = (100 * by_task["kept"] / by_task["answered"].clip(lower=1)).round(1)
by_task["benchmark_accuracy_%"] = by_task["task"].map(BENCHMARK_ACCURACY)
by_task.to_csv(OUTPUT_DIR / "by_task.csv", index=False)
print(by_task.to_string(index=False))
complete = len(rows) >= len(TODO_ALL)
expected = 700 <= len(kept_rows) <= 900
print(f"\n{len(kept_rows):,} kept of {len(rows):,} answered. Expected about 700-900: "
      + ("n/a (smoke test)" if SMOKE_TEST else ("met" if expected else "NOT met")))
config = {"task": "L2 step 1c: the options rerun", "smoke_test": SMOKE_TEST, "teacher": TEACHER_MODEL_ID, "image_size": "full (1036 px)",
          "decoding": "greedy", "prompts": "the benchmark's, with the options", "questions": len(TODO_ALL), "answered": len(rows),
          "kept": len(kept_rows), "complete": complete, "held_out_every": HELD_OUT_EVERY, "hours_this_session": round(RUN_HOURS, 2)}
(OUTPUT_DIR / "config.json").write_text(json.dumps(config, indent=1), encoding="utf-8")
(OUTPUT_DIR / "README.md").write_text("\n".join([
    "# L2 step 1c: the options rerun" + (" (SMOKE TEST)" if SMOKE_TEST else ""), "",
    f"The 7B `{TEACHER_MODEL_ID}` answered L2 step 1's counting and relational questions again with the benchmark's prompts and "
    "options, at full size, greedy. An answer is kept when its letters equal the label's. "
    f"{len(kept_rows):,} kept of {len(rows):,}; held-out after-photos left out.", "",
    "| " + " | ".join(by_task.columns) + " |", "|" + "---|" * len(by_task.columns),
    *["| " + " | ".join(str(v) for v in r) + " |" for r in by_task.itertuples(index=False)], "",
    "Expected about 700-900 kept (fixed before the run): " + ("n/a (smoke test)" if SMOKE_TEST else ("met" if expected else "**not met**")) + ".", "",
    "## Files", "",
    "- `options_kept.jsonl`: per kept example its entry index, task, the benchmark prompt (with `<image>` where each picture goes), "
    "the image paths in order, and the answer letter. Option (c)'s training adds these to L2 step 1's data.",
    "- `options_progress.jsonl` (every answer), `by_task.csv`, `config.json`.",
]) + "\n", encoding="utf-8")
print(f"✓ Wrote {OUTPUT_DIR}: options_kept.jsonl, options_progress.jsonl, by_task.csv, README.md, config.json")
